# Your first Claude Skill — a Methods section drafter

**Companion notebook** for the rendered note at
👉 [tensoromics.com/notes/your-first-claude-skill-methods-drafter](https://tensoromics.com/notes/your-first-claude-skill-methods-drafter)

**What this notebook does**

1. Writes a `SKILL.md` file you can copy into `~/.claude/skills/methods-drafter/` to use as a real Claude Code Skill.
2. Demonstrates the same rules via the Anthropic Python SDK — using `SKILL.md` body as a system prompt — so you can verify the behaviour without leaving Jupyter.
3. Runs three worked examples: a clean RNA-seq DEG draft, an scRNA-seq draft with deliberate gaps (to show the `[Missing: ...]` anti-hallucination behaviour), and a multi-section project.

> **Editing workflow.** This notebook is the runnable source. The rendered note on the website mirrors its prose and code. Port changes here to `tensoromics-web/content/notes/your-first-claude-skill-methods-drafter.mdx` to keep them in sync.

## 0. Setup

In [ ]:
%pip install --quiet anthropic

In [ ]:
import os
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before continuing.'

## 1. Write the SKILL.md file

Run this cell. It writes the Skill into `~/.claude/skills/methods-drafter/SKILL.md` so Claude Code can pick it up.

If you don't have Claude Code installed, the file is still written for later — and the rest of the notebook uses the same `SKILL_BODY` string as a system prompt against the Anthropic API, so you can verify the rules work.

In [ ]:
from pathlib import Path

SKILL_BODY = '''---
name: methods-drafter
description: Use when the user pastes lab-notebook bullets, README snippets, script header comments, or analysis shorthand and asks for a Methods section. Activates for any "draft methods" / "write up the methods" request.
---

# Methods drafter

You are a senior computational biologist drafting Methods sections for top-tier journals (Nature, Cell, Genome Biology).

When the user gives you shorthand notes — bullets, README excerpts, Snakefile params, script comments, an email thread — rewrite them as a single flowing Methods paragraph (or one paragraph per sub-analysis if multiple are present).

## Style rules

- Past tense, passive voice (standard for Methods)
- Include every software name, version, and parameter the user mentioned
- Make implicit details explicit ("BH" → "Benjamini–Hochberg procedure")
- Expand abbreviations on first use (DEG, FDR, PCA, GSEA)
- No editorialising, no praise, no "we believe", no "interestingly"
- No bullet points in output — flowing prose only

## Anti-hallucination (load-bearing)

If the user's notes omit a normally-required detail (reference genome version, statistical test, sample size, library prep, alignment parameters), note it at the end as `[Missing: ...]`. **Never invent values.**

## Multi-paragraph form

If the user gives you several sub-analyses, draft each as its own paragraph under a `### <subheading>`. The reader should be able to copy the whole block straight into a manuscript skeleton.

## Output format

Return ONLY the Methods paragraph (or paragraphs + subheadings) plus the `[Missing: ...]` line if anything is missing. No preamble, no explanation, no "here is your Methods section."
'''

skill_dir = Path.home() / '.claude' / 'skills' / 'methods-drafter'
skill_dir.mkdir(parents=True, exist_ok=True)
skill_path = skill_dir / 'SKILL.md'
skill_path.write_text(SKILL_BODY)

print(f'Wrote {skill_path}')
print(f'Size: {len(SKILL_BODY)} chars')

## 2. Use the same rules via the Anthropic API

Strip the YAML frontmatter from `SKILL_BODY` and use the remainder as a system prompt. This is exactly the behaviour Claude Code applies when the Skill auto-loads.

In [ ]:
from anthropic import Anthropic

client = Anthropic()
MODEL = 'claude-opus-4-7'

# Strip the YAML frontmatter — keep only the instructional body.
def skill_to_system_prompt(skill_md: str) -> str:
    parts = skill_md.split('---', 2)
    # parts = ['', 'frontmatter yaml', '\n# Methods drafter\n...']
    return parts[2].strip() if len(parts) >= 3 else skill_md

SYSTEM_PROMPT = skill_to_system_prompt(SKILL_BODY)

def draft_methods(user_message: str) -> str:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1500,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_message}],
    )
    return resp.content[0].text.strip()

## 3. Worked example 1 — bulk RNA-seq DEG analysis

A clean set of notes. The Skill should produce a complete Methods paragraph with no `[Missing: ...]` line.

In [ ]:
deg_notes = '''
RNA-seq DEG analysis — 2026-04-15
- 24 samples, 6 conditions, 4 reps each
- aligned with STAR v2.7.11 to GRCh38 + GENCODE v45
- gene-level expression with RSEM (v1.3.3)
- DESeq2 v1.42, alpha=0.05, BH for multiple testing
- pre-filtered: drop genes with sum<10 across all samples
- sequencing batch as covariate
- 3 outliers dropped on PCA (PC1 > 2 SD)

Draft the Methods section for this.
'''

print(draft_methods(deg_notes))

**Expected** (wording varies, structure does not):

A single flowing paragraph mentioning STAR v2.7.11, GRCh38, GENCODE v45, RSEM (v1.3.3), the filter step, three outliers excluded on PC1 > 2 SD, DESeq2 v1.42, batch covariate, BH-adjusted FDR < 0.05. Past tense, passive voice, no bullets. No `[Missing: ...]` line — the notes were complete.

## 4. Worked example 2 — scRNA-seq with deliberate gaps

We omit the reference genome, QC thresholds, normalisation method, and integration strategy on purpose. The Skill should write a clean paragraph for what's there AND flag what's missing.

**This is the load-bearing test.** If the model invents a reference genome version, the Skill prompt isn't strict enough — that's a real failure mode worth catching now.

In [ ]:
scrna_notes = '''
scRNA-seq:
- 10x Genomics Chromium
- Cell Ranger v8.0
- Seurat v5.1 clustering, resolution 0.6
- 8 samples, 2 donors x 4 timepoints

Methods section please.
'''

print(draft_methods(scrna_notes))

**Expected**:

1. A paragraph that mentions ONLY what was provided (Chromium, Cell Ranger v8.0, Seurat v5.1, resolution 0.6, 8 samples × 2 donors × 4 timepoints).
2. A `[Missing: ...]` line listing reference genome version, Chromium chemistry, QC thresholds, normalisation, integration, cell-type annotation strategy.

**If the output invents a reference genome version**, the Skill needs tightening — that's a hallucination the rule should have caught.

## 5. Worked example 3 — multi-section project

Three sub-analyses in one input. The Skill should produce three named subsections (`### Heading`), each a clean Methods paragraph.

In [ ]:
multi_notes = '''
Three sub-analyses. Draft a full Methods section with subsections.

Section 1 — Alignment and quantification
- aligned with STAR v2.7.11 to GRCh38 + GENCODE v45
- RSEM (v1.3.3) for gene-level expression
- pre-filter: drop genes with sum<10 across all samples

Section 2 — Differential expression
- DESeq2 v1.42, alpha=0.05, BH for multiple testing
- batch covariate; 3 outliers dropped on PCA (PC1 > 2 SD)
- 24 samples, 6 conditions, 4 reps each

Section 3 — Pathway enrichment
- GSEA via clusterProfiler v4.10
- MSigDB Hallmark v2023.2.Hs
- 10,000 permutations, FDR < 0.05
'''

print(draft_methods(multi_notes))

**Expected**: three `### <Subheading>` blocks, each a clean Methods paragraph in journal style. Should be drop-in usable as a manuscript skeleton.

## 6. Verify it as a real Claude Code Skill

If you have Claude Code installed locally, the file is already in place at `~/.claude/skills/methods-drafter/SKILL.md`. Open a new Claude Code session in any directory, paste this:

> *Here are my analysis notes — please draft the Methods section.*
>
> *RNA-seq DEG analysis — 2026-04-15*
> *- 24 samples, 6 conditions, 4 reps each*
> *- aligned with STAR v2.7.11 to GRCh38 + GENCODE v45*
> *...*

Claude Code should auto-load the Skill (the description matches the request) and reply in the Skill's voice. If you want to confirm the Skill loaded, ask Claude `/skills` — `methods-drafter` should appear in the list of loaded Skills for the session.

## 7. Where to take this next

- **Per-journal variants.** Make `methods-drafter-nature`, `methods-drafter-cell`, etc. — each with its target journal's house-style rules. Claude picks based on the description.
- **Two-pass polish.** Drafter Skill writes; a separate `methods-polisher` Skill tightens redundancy without losing detail.
- **Pair with a DEG-reviewer Skill** (next note in the series) — one Skill writes up your analysis, the other reviews the code it came from.
- **Project-level Skill.** Move the file to `<your-repo>/.claude/skills/methods-drafter/SKILL.md` so the Skill ships with the codebase and your collaborators get the same Methods style automatically.